In [ ]:
%load_ext autoreload
%autoreload 2

import itertools
import glob
import json
import os
import pickle
import sys

import dotenv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tqdm

from matplotlib.colors import BoundaryNorm, ListedColormap, LogNorm
import matplotlib.lines as mlines
from matplotlib.ticker import LogFormatterMathtext

dotenv.load_dotenv()
PROJECT_ROOT = os.environ["PROJECT_ROOT"]
sys.path.append(PROJECT_ROOT)

import mpl.experiments.utils as OPE_utils
import mpl.utils.argparsing as argparsing

In [ ]:
arrays_root = f"{PROJECT_ROOT}/data/arrays"
os.makedirs(arrays_root, exist_ok=True)
datasets = ["mslr", "yahoo"]

In [ ]:
for ds in datasets:
    for fold in tqdm.tqdm(range(1, 6)):
        config_path = f"{PROJECT_ROOT}/configs/models/train_RQ3_{ds}_logging.json"
        model_config = json.load(open(config_path))
        model_config["FOLD"] = fold
        model_config = argparsing.parse_nested_dict(model_config)
        data = model_config["data"]()
        for subset in ["validation", "test"]:
            mask, Y = [], []
            for x, m, y in iter(data[subset]):
                Y.append(y.numpy()), mask.append(m.numpy())
            Y, mask = np.concatenate(Y, axis=0)[:, :, 0], np.concatenate(mask, axis=0)
            os.makedirs(f"{arrays_root}/{ds}/{fold}/{subset}", exist_ok=True)
            np.save(f"{arrays_root}/{ds}/{fold}/{subset}/y.npy", Y)
            np.save(f"{arrays_root}/{ds}/{fold}/{subset}/mask.npy", mask)

In [ ]:
prop_root = f"{PROJECT_ROOT}/experiments/propensities"
click_root = f"{PROJECT_ROOT}/experiments/clicks"
y_root = f"{PROJECT_ROOT}/local_data"
ppaths = glob.glob(f"{prop_root}/**/RQ3_*/**/propensities.npy", recursive=True)
tpaths = glob.glob(f"{prop_root}/**/RQ3_*/**/time.npy", recursive=True)
cpaths = [
    x
    for ds in datasets
    for j in range(1, 6)
    for x in glob.glob(
        f"{PROJECT_ROOT}/experiments/clicks/{ds}/{j}/**/clicks.npy", recursive=True
    )
]
ypaths = glob.glob(f"{arrays_root}/**/y.npy", recursive=True)
mpaths = glob.glob(f"{arrays_root}/**/mask.npy", recursive=True)

In [ ]:
ps = {k: np.load(k) for k in tqdm.tqdm(ppaths)}
ts = {k: np.load(k) for k in tqdm.tqdm(tpaths)}
cs = {k: np.load(k) for k in tqdm.tqdm(cpaths)}
ys = {k: np.load(k) for k in tqdm.tqdm(ypaths)}
ms = {k: np.load(k) for k in tqdm.tqdm(mpaths)}

In [ ]:
mappings = {
    6: ["dataset", "FOLD", "model", "method", "pol", "filename"],  # IPM
}
cmappings = {
    5: ["dataset", "FOLD", "model", "clicks", "filename"],  # IPM
}
ymappings = {4: ["dataset", "FOLD", "pol:subset"]}

In [ ]:
records = []
for (pp, p), (tp, t) in tqdm.tqdm(zip(ps.items(), ts.items())):
    records.append(
        {
            **OPE_utils.flatten_dict(
                OPE_utils.path_to_params(pp, prop_root, mappings), sep=":"
            ),
            "p": p,
            "time": t.item(),
        }
    )
df = pd.DataFrame.from_records(records)
df["model"] = df["model"].apply(lambda x: x.split("_")[-1])

In [ ]:
records = []
for (yp, y), (mp, m) in tqdm.tqdm(zip(ys.items(), ms.items())):
    records.append(
        {
            **OPE_utils.flatten_dict(
                OPE_utils.path_to_params(yp, arrays_root, ymappings), sep=":"
            ),
            "y": y,
            "mask": m,
        }
    )
ydf = pd.DataFrame.from_records(records)

In [ ]:
oracles = df[df["pol:N"] == 10000001.0]
df = df[df["pol:N"] != 10000001.0]
oracles = oracles.merge(ydf, on=["dataset", "FOLD", "pol:subset"], how="inner")

In [ ]:
records = []
for path, c in tqdm.tqdm(cs.items()):
    records.append(
        {
            **OPE_utils.flatten_dict(
                OPE_utils.path_to_params(path, click_root, cmappings), sep=":"
            ),
            "c": c,
        }
    )
cdf = pd.DataFrame.from_records(records)
cdf["model"] = cdf["model"].apply(lambda x: x.split("_")[-1])

In [ ]:
cdf = cdf.rename({"clicks:subset": "pol:subset"}, axis=1)
oracles["pol:l"] = 0
df["pol:l"] = df["pol:l"].fillna(0)

In [ ]:
metric = "MSE"


def IPS_PBM(
    clicks,
    pol_log,
    pol_tgt,
    y,
    pol_tgt_oracle,
    mask,
    pos_bias,
    clip=0.0,
    metric="MSE",
    pol_log_oracle=None,
):
    pol_log_oracle = (
        pol_log_oracle if pol_log_oracle is not None else np.ones_like(pol_tgt_oracle)
    )
    inputs = clicks, pol_log, pol_tgt, y, pol_tgt_oracle, mask, pol_log_oracle
    minsize = np.min([k.shape[0] for k in inputs])
    n_docs = mask.shape[1]
    clicks, pol_log, pol_tgt, y, pol_tgt_oracle, mask, pol_log_oracle = [
        k[:minsize].reshape(minsize, n_docs, -1) for k in inputs
    ]
    n = mask.any(axis=(-1, -2)).sum()
    w = (pol_tgt * pos_bias).sum(-1) / np.maximum((pol_log * pos_bias).sum(-1), clip)
    y_pred = clicks.sum(-1) * w
    y_true = y * pol_tgt_oracle * pos_bias
    y_pred, y_true = np.where(mask[:, :, 0], y_pred, 0), np.where(mask, y_true, 0)
    y_pred, y_true = y_pred.sum((-1)), y_true.sum((-2, -1))
    metric = {"MSE": lambda x, y: (x - y) ** 2}[metric]
    error = metric(y_pred, y_true).sum() / n
    return error, y_pred, y_true


def IPS_IPM(
    clicks,
    pol_log,
    pol_tgt,
    y,
    pol_tgt_oracle,
    mask,
    pos_bias,
    clip=0.0,
    metric="MSE",
    pol_log_oracle=None,
    K=10,
    _K=10,
    _bs=None,
):
    pol_log_oracle = (
        pol_log_oracle if pol_log_oracle is not None else np.ones_like(pol_tgt_oracle)
    )
    inputs = clicks, pol_log, pol_tgt, y, pol_tgt_oracle, mask, pol_log_oracle
    if _bs is not None:
        inputs = [x[:_bs] for x in inputs]
    minsize = np.min([k.shape[0] for k in inputs])
    n_docs = mask.shape[1]
    clicks, pol_log, pol_tgt, y, pol_tgt_oracle, mask, pol_log_oracle = [
        k[:minsize].reshape(minsize, n_docs, -1) for k in inputs
    ]
    n = mask.any(axis=(-1, -2)).sum()
    y_pred = clicks * pol_tgt / np.maximum(pol_log, clip)
    y_true = y * pol_tgt_oracle * pos_bias
    m2 = mask & (mask.sum(1, keepdims=True) > np.arange(K))
    y_pred, y_true = np.where(m2, y_pred, 0), np.where(m2, y_true, 0)
    y_pred, y_true = y_pred[..., :_K].sum((-2, -1)), y_true[..., :_K].sum((-2, -1))
    metric = {"MSE": lambda x, y: (x - y) ** 2}[metric]
    error = metric(y_pred, y_true).sum() / n
    return error, y_pred, y_true

In [ ]:
models = ["logging", "target"]
log_row_cols = ["dataset", "FOLD", "model", "pol:N", "pol:l", "pol:subset", "method"]
columns = log_row_cols + ["tgt_pol", "click:N", "clip", "estimator", "value"]
datasets = ["yahoo", "mslr"]
subsets = ["validation", "test"]
estimators = ["IPS_IPM", "IPS_PBM"]

result_output_root = os.path.join(PROJECT_ROOT, "experiments", "results")

In [ ]:
oracle_cols = ["FOLD"]
same_cols = oracle_cols + ["method", "pol:N"]
click_cols = ["FOLD"]
click_Ns = [100, 1000, 10000, 100000, 1000000, 10000000]
clips = 10 ** -np.arange(3, 9.0)
pos_bias = np.log2(np.arange(2, 12)) ** -1
stop = False

for subset in subsets:
    for dataset in datasets:
        subset_df = df[(df["dataset"] == dataset) & (df["pol:subset"] == subset)]
        oracle_df = oracles[
            (oracles["dataset"] == dataset) & (oracles["pol:subset"] == subset)
        ]
        click_df = cdf[(cdf["dataset"] == dataset) & (cdf["pol:subset"] == subset)]
        for estimator in estimators:
            estimator_fn = {
                "IPS_IPM": lambda *args, **kwargs: IPS_IPM(*args, **kwargs),
                "IPS_PBM": lambda *args, **kwargs: IPS_PBM(*args, **kwargs),
            }[estimator]
            results = {}
            for idx in tqdm.tqdm(subset_df[subset_df.model == "logging"].index):
                log_row = subset_df.loc[idx]
                log_pol = log_row["model"]
                for tgt_pol in models:
                    if tgt_pol == log_pol:
                        continue
                    tgt_row_mask = (subset_df[same_cols] == log_row[same_cols]).all(
                        1
                    ) & (subset_df["model"] == tgt_pol)
                    tgt_row = subset_df[tgt_row_mask].iloc[0]
                    oracle_row_mask = (
                        oracle_df[oracle_cols] == log_row[oracle_cols]
                    ).all(1) & (oracle_df["model"] == tgt_pol)
                    oracle_row = oracle_df[oracle_row_mask].iloc[0]
                    for click_N in click_Ns:
                        click_row_mask = (
                            (click_df[click_cols] == log_row[click_cols]).all(1)
                            & (click_df["model"] == log_pol)
                            & (click_df["clicks:N"] == click_N)
                        )
                        click_row = click_df[click_row_mask].iloc[0]
                        clicks, pol_log, pol_tgt, y, pol_tgt_oracle, mask = (
                            click_row["c"].astype(np.float32),
                            log_row["p"].astype(np.float32),
                            tgt_row["p"].astype(np.float32),
                            oracle_row["y"].astype(np.float32),
                            oracle_row["p"].astype(np.float32),
                            oracle_row["mask"],
                        )
                        for clip in clips:
                            key = (
                                *log_row[log_row_cols].values,
                                tgt_pol,
                                click_N,
                                clip,
                                estimator,
                            )
                            pred = estimator_fn(
                                clicks,
                                pol_log,
                                pol_tgt,
                                y,
                                pol_tgt_oracle,
                                mask,
                                pos_bias,
                                clip=clip,
                                metric=metric,
                            )[0]
                            assert key not in results
                            results[key] = pred

            result_path = os.path.join(
                result_output_root, f"{dataset}_{subset}_{estimator}.pkl"
            )
            res_df = pd.DataFrame(
                [(*k, v) for k, v in results.items()], columns=columns
            )
            res_df.to_pickle(result_path)
            del res_df

In [ ]:
# Visualization

In [ ]:
figure_dir = f"{PROJECT_ROOT}/experiments/results/figures"
os.makedirs(figure_dir, exist_ok=True)

In [ ]:
# MSE
for dataset, estimator in itertools.product(datasets, estimators):
    # Find best validation clipping value
    validation_path, test_path = (
        os.path.join(result_output_root, f"{dataset}_validation_{estimator}.pkl"),
        os.path.join(result_output_root, f"{dataset}_test_{estimator}.pkl"),
    )
    validation_df, test_df = (
        pickle.load(open(validation_path, "rb")),
        pickle.load(open(test_path, "rb")),
    )
    validation_test_groupby_cols = ["dataset", "FOLD", "pol:N", "method", "click:N"]
    min_idx = validation_df.groupby(validation_test_groupby_cols)["value"].idxmin()
    validation_df = validation_df.loc[min_idx]
    res_df = validation_df[validation_test_groupby_cols + ["clip"]].merge(
        test_df, how="left"
    )
    tdf = test_res_df = res_df[~res_df["value"].isna()]

    tdf = tdf.sort_values("method").reset_index()

    log_row_cols = [
        "dataset",
        "FOLD",
        "model",
        "pol:N",
        "pol:l",
        "pol:subset",
        "method",
    ]
    columns = log_row_cols + ["tgt_pol", "click:N", "clip", "value"]

    maxs = tdf.groupby(
        [x for x in columns if x not in ["value", "FOLD", "clip", "pol:l"]]
    )["value"].apply(np.max)
    mins = tdf.groupby(
        [x for x in columns if x not in ["value", "FOLD", "clip", "pol:l"]]
    )["value"].apply(np.min)
    tdf = (
        tdf.groupby(
            [x for x in columns if x not in ["value", "FOLD", "clip", "pol:l"]]
        )["value"]
        .apply(np.mean)
        .reset_index()
    )
    tdf = tdf.rename({"value": "mean"}, axis=1)
    tdf["max"] = maxs.values
    tdf["min"] = mins.values

    fig, axes = plt.subplots(1, 1, figsize=(1.5, 5))
    linestyles = {"MPL": "-", "MC": "--"}
    markers = {"MPL": "v", "MC": "o"}
    masks = [(tdf["tgt_pol"] == "target") & (tdf["model"] == f"logging")]
    colors = {
        ("MC", 1000): "C0",
        ("MC", 10000): "C1",
        ("MC", 100000): "C2",
        ("MC", 1000000): "C3",
        ("MC", 10000000): "C4",
        ("MPL", 50): "C5",
        ("MPL", 1000): "C6",
    }
    labels = {
        ("MC", 1000): r"MC:$10^3$",
        ("MC", 10000): r"MC:$10^4$",
        ("MC", 100000): r"MC:$10^5$",
        ("MC", 1000000): r"MC:$10^6$",
        ("MC", 10000000): r"MC:$10^7$",
        ("MPL", 50): r"MPL:$50$",
        ("MPL", 1000): r"MPL:$10^3$",
    }

    for ax, mask in zip(axes.flatten() if hasattr(axes, "flatten") else [axes], masks):
        grouping = tdf[mask].groupby(["model", "tgt_pol", "method", "pol:N"])
        for k, group in grouping:
            method = k[2]
            if method == "MPL":
                if k[3] not in [50, 1000]:
                    continue
            pols = k[0], k[1]
            ax.plot(
                group["click:N"],
                group["mean"],
                label=k,
                marker=markers[method],
                linestyle=linestyles[method],
                color=colors[(method, k[3])],
            )
            ax.fill_between(group["click:N"], group["min"], group["max"], alpha=0.3)
        ax.set_yscale("log")
        ax.set_xscale("log")

        # Set labels
        xticks = [10**i for i in range(3, 8, 2)]
        ax.set_xticks(xticks)
        ax.xaxis.set_major_formatter(LogFormatterMathtext(base=10.0))

    figure_path = f"{figure_dir}/OPE_{dataset}_{estimator}.pdf"
    print(figure_path)
    fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0)

In [ ]:
# MSE Legend
for dataset in datasets:
    markers = {
        ("mslr", "MPL"): "v",
        ("mslr", "MC"): "o",
        ("yahoo", "MPL"): "^",
        ("yahoo", "MC"): "*",
    }

    legend_info = []
    legend_info.append(
        {
            "linestyle": "-",
            "color": "r",
            "marker": ".",
            "markersize": 0,
            "markevery": 1,  # this should be higher if you have more data
            "fillstyle": "full",
            "label": {"mslr": "MSLR:", "yahoo": "Yahoo:"}[dataset],
            "linewidth": 0,
        }
    )
    for i in [3, 0, 4, 1, 5, 2, 6]:
        k = list(colors)[i]
        legend_info.append(
            {
                "linestyle": "-",
                "color": colors[k],
                "marker": markers[(dataset, k[0])],
                "markersize": 12,
                "markevery": 1,  # this should be higher if you have more data
                "fillstyle": "full",
                "label": labels[k],
                "linewidth": None,
            }
        )

    ncol = 4
    figlegend = plt.figure(figsize=(0.5, 0.5))
    leg = figlegend.legend(
        handles=[mlines.Line2D([], [], **l) for l in legend_info],
        fontsize=18,
        loc="center",
        ncol=ncol,
        frameon=False,
        borderaxespad=0,
        borderpad=0.3,
        labelspacing=0.0,
        columnspacing=1,
        handlelength=0.1,
    )
    texts = leg.get_texts()
    texts[0].set_x(texts[0].get_position()[0] - 21)
    figlegend.savefig(
        f"{figure_dir}/legend_{dataset}.pdf", bbox_inches="tight", pad_inches=0.0
    )

In [ ]:
# Optimal clipping value
datasets, estimators, methods = ["mslr", "yahoo"], ["IPS_IPM", "IPS_PBM"], ["MPL", "MC"]
for dataset, estimator, method in itertools.product(datasets, estimators, methods):
    validation_result_path = os.path.join(
        result_output_root, f"{dataset}_validation_{estimator}.pkl"
    )
    a = pickle.load(open(validation_result_path, "rb"))
    c = a

    log_row_cols = [
        "dataset",
        "FOLD",
        "model",
        "pol:N",
        "pol:l",
        "pol:subset",
        "method",
    ]
    columns = log_row_cols + ["tgt_pol", "click:N", "clip", "value"]
    new_idx = (
        c[c["pol:l"] != 1e-3]
        .groupby([x for x in columns if x not in ["value", "clip", "pol:l"]])["value"]
        .apply(lambda x: x.nsmallest(1).index[-1])
    )
    c = c.loc[new_idx]
    c = c.sort_values("method").reset_index()

    z = c[
        (c.method == method)
        & (c.FOLD == "1")
        & (c.tgt_pol == "target")
        & (c.model == "logging")
    ]
    heatmap_data = z[["pol:N", "click:N", "clip"]].pivot(
        index="pol:N", columns="click:N", values="clip"
    )
    fig, axes = plt.subplots(1, 1, figsize=(1.3, 1.1))
    ax = axes
    cmap = sns.color_palette("rocket", n_colors=6)
    cmap = ListedColormap(cmap)
    norm = LogNorm(vmin=1e-8, vmax=1e-4)
    ax = sns.heatmap(heatmap_data, cmap=cmap, cbar=False, norm=norm, ax=ax)
    x_values = np.logspace(2, 7, 6)

    ax.set_xticks(
        [x for x in ax.get_xticks()[1::2]] + [x for x in ax.get_xticks()[::2]]
    )
    ax.set_xticklabels(
        [rf"$10^{{{int(np.log10(v))}}}$" for v in x_values[1::2]] + [""] * 3, rotation=0
    )
    y_values = np.logspace(3, 7, 5)
    ax.set_yticks(ax.get_yticks())

    if method == "MC":
        ax.set_yticklabels([rf"$10^{{{int(np.log10(v))}}}$" for v in y_values])
    else:
        ax.set_yticklabels([50, 100, 200, 500, 1000])
    ax.set_yticklabels([])
    ax.set_xlabel(""), ax.set_ylabel("")

    figure_path = (
        f"{figure_dir}/OPE_{dataset}_clipping_heatmap_{method}_{estimator}.pdf"
    )
    fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0.01)

In [ ]:
# Optimal clipping legend
values = [x for x in [2, 3, 4, 5, 6, 7]]
boundaries = [x - 0.5 for x in values + [values[-1] + 1]]

cmap = sns.color_palette("rocket", n_colors=len(values))
cmap = ListedColormap(cmap)
norm = BoundaryNorm(boundaries=boundaries, ncolors=len(values))

fig, cax = plt.subplots(figsize=(4, 1))

cax.set_visible(False)

cbar = fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=cax,
    orientation="horizontal",
    ticks=values,
)

cbar.ax.set_xticklabels([rf"$10^{{{v - 10}}}$" for v in values])
cbar.ax.tick_params(size=0)
figure_path = f"{figure_dir}/OPE_clipping_heatmap_legend.pdf"
print(figure_path)
fig.savefig(figure_path, bbox_inches="tight", dpi=300, pad_inches=0)